In [1]:
# ============================================================
# Spam Filter by Dr. Arpit Yadav
# Simple Reflex AI Agent using LangGraph + Groq + Serper + Gradio
# ============================================================

# =========================
# 1. Install dependencies
# =========================
!pip -q install langgraph langchain langchain-groq gradio requests pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.1 MB/s eta 0:00:00


In [2]:

# =========================
# 2. Import all necessary libraries
# =========================
import os
import re
import json
import time
import requests
from typing import TypedDict, List, Dict, Any

import gradio as gr

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

In [3]:
# =========================
# 3. Give Groq API and Serper API
# =========================
# In Colab, run this cell and paste your keys when prompted if needed.
# Or set directly:
# os.environ["GROQ_API_KEY"] = "your_groq_api_key"
# os.environ["SERPER_API_KEY"] = "your_serper_api_key"

if "GROQ_API_KEY" not in os.environ:
    try:
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("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")
    except Exception:
        pass

if "SERPER_API_KEY" not in os.environ:
    try:
        from getpass import getpass
        os.environ["SERPER_API_KEY"] = getpass("34cf9bb4a6965f4ad9bcce86f48f5d693048ac09")
    except Exception:
        pass

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
SERPER_API_KEY = os.getenv("SERPER_API_KEY", "")

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··········
34cf9bb4a6965f4ad9bcce86f48f5d693048ac09··········


In [4]:

# =========================
# 4. Select Model from Groq
#    use llama-3.1-8b-instant
# =========================
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY
)


In [5]:


# =========================
# 5. Simple Reflex Agent Logic
# =========================
# The agent uses ONLY current input and fixed rules.
# No memory, no history, no learning.

SPAM_KEYWORDS = [
    "lottery", "winner", "won", "free money", "claim now", "urgent action",
    "verify your account", "account suspended", "click here", "limited time",
    "congratulations", "prize", "bitcoin", "crypto giveaway", "investment guaranteed",
    "password reset immediately", "bank update", "act now", "exclusive deal",
    "100% free", "risk-free", "earn money fast", "double your income",
    "gift card", "wire transfer", "refund pending", "tax refund"
]

PROMO_KEYWORDS = [
    "discount", "sale", "offer", "coupon", "deal", "subscribe",
    "newsletter", "shop now", "buy now", "special promotion", "launch offer"
]

PHISHING_PATTERNS = [
    r"verify\s+your\s+account",
    r"reset\s+your\s+password",
    r"bank\s+account",
    r"login\s+immediately",
    r"security\s+alert",
    r"suspended",
    r"unusual\s+activity",
    r"confirm\s+identity"
]

URL_REGEX = r"(https?://[^\s]+|www\.[^\s]+)"
EMAIL_REGEX = r"[\w\.-]+@[\w\.-]+\.\w+"

SAFE_DOMAINS = [
    "google.com", "microsoft.com", "openai.com", "github.com",
    "linkedin.com", "amazon.com", "coursera.org"
]

BLOCKED_TLDS = [".xyz", ".top", ".click", ".work", ".gq", ".tk", ".buzz"]


class SpamState(TypedDict, total=False):
    email_text: str
    sender: str
    subject: str
    search_mode: str
    strictness: str
    category: str
    confidence: float
    score: int
    matched_rules: List[str]
    extracted_urls: List[str]
    web_evidence: str
    explanation: str
    suggested_action: str
    final_report: str


# =========================
# Utility functions
# =========================
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def extract_urls(text: str) -> List[str]:
    if not text:
        return []
    return re.findall(URL_REGEX, text)

def extract_emails(text: str) -> List[str]:
    if not text:
        return []
    return re.findall(EMAIL_REGEX, text)

def domain_from_url(url: str) -> str:
    url = url.lower().replace("https://", "").replace("http://", "").replace("www.", "")
    return url.split("/")[0]

def looks_obfuscated(text: str) -> bool:
    patterns = [
        r"fr[e3]{2}\s*m[o0]ney",
        r"cl[i1]ck\s*h[e3]re",
        r"w[i1]nn[e3]r",
        r"pa[s5]{2}word",
        r"v[e3]r[i1]fy"
    ]
    return any(re.search(p, text.lower()) for p in patterns)

def excessive_punctuation(text: str) -> bool:
    return bool(re.search(r"[!]{3,}|[$]{3,}|[?]{3,}", text))

def excessive_caps(text: str) -> bool:
    words = text.split()
    caps_words = [w for w in words if len(w) >= 4 and w.isupper()]
    return len(caps_words) >= 4

def strictness_weight(strictness: str) -> int:
    mapping = {
        "Low": 12,
        "Medium": 9,
        "High": 6
    }
    return mapping.get(strictness, 9)


# =========================
# Serper web search
# =========================
def serper_search(query: str, num_results: int = 3) -> str:
    """
    Optional enrichment only.
    Not used to make final classification.
    """
    if not SERPER_API_KEY:
        return "Serper API key not available."

    url = "https://google.serper.dev/search"
    payload = json.dumps({
        "q": query,
        "num": num_results
    })
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(url, headers=headers, data=payload, timeout=20)
        response.raise_for_status()
        data = response.json()

        snippets = []
        for item in data.get("organic", [])[:num_results]:
            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")
            snippets.append(f"- {title} | {link}\n  {snippet}")

        if not snippets:
            return "No notable web evidence found."
        return "\n".join(snippets)

    except Exception as e:
        return f"Serper search failed: {str(e)}"


# =========================
# LangGraph nodes
# =========================
def preprocess_node(state: SpamState) -> SpamState:
    email_text = clean_text(state.get("email_text", ""))
    sender = clean_text(state.get("sender", ""))
    subject = clean_text(state.get("subject", ""))

    merged = f"Subject: {subject}\nSender: {sender}\nBody: {email_text}"
    urls = extract_urls(merged)

    return {
        **state,
        "email_text": merged,
        "extracted_urls": urls,
        "matched_rules": [],
        "score": 0
    }


def reflex_rule_engine_node(state: SpamState) -> SpamState:
    """
    Core Simple Reflex Logic:
    Uses only current email input and fixed rules.
    """
    text = state["email_text"].lower()
    urls = state.get("extracted_urls", [])
    strictness = state.get("strictness", "Medium")

    score = 0
    matched_rules = []

    keyword_weight = strictness_weight(strictness)

    # Rule 1: spam keywords
    for kw in SPAM_KEYWORDS:
        if kw in text:
            score += keyword_weight
            matched_rules.append(f"Spam keyword detected: '{kw}'")

    # Rule 2: promotional keywords
    promo_hits = 0
    for kw in PROMO_KEYWORDS:
        if kw in text:
            promo_hits += 1
    if promo_hits >= 2:
        score += 12
        matched_rules.append(f"Promotional language detected: {promo_hits} promo terms")

    # Rule 3: phishing patterns
    for pattern in PHISHING_PATTERNS:
        if re.search(pattern, text):
            score += 18
            matched_rules.append(f"Phishing pattern detected: /{pattern}/")

    # Rule 4: suspicious URLs
    for u in urls:
        domain = domain_from_url(u)
        if any(domain.endswith(tld) for tld in BLOCKED_TLDS):
            score += 20
            matched_rules.append(f"Suspicious top-level domain: {domain}")
        elif domain not in SAFE_DOMAINS and "." in domain:
            score += 6
            matched_rules.append(f"Unknown external domain found: {domain}")

    # Rule 5: obfuscated words
    if looks_obfuscated(text):
        score += 14
        matched_rules.append("Obfuscated spam/phishing words detected")

    # Rule 6: punctuation abuse
    if excessive_punctuation(text):
        score += 10
        matched_rules.append("Excessive punctuation detected")

    # Rule 7: all-caps style
    if excessive_caps(state["email_text"]):
        score += 8
        matched_rules.append("Excessive all-caps words detected")

    # Rule 8: too many links
    if len(urls) >= 3:
        score += 12
        matched_rules.append(f"Too many links detected: {len(urls)} URLs")

    # Rule 9: sender mismatch hints
    sender = state.get("sender", "").lower()
    body_emails = extract_emails(state["email_text"].lower())
    if sender and body_emails:
        sender_domain = sender.split("@")[-1] if "@" in sender else sender
        mismatch = any((e.split("@")[-1] != sender_domain) for e in body_emails if "@" in e)
        if mismatch:
            score += 10
            matched_rules.append("Sender/body domain mismatch hint detected")

    # Final category based on score
    if score >= 45:
        category = "Spam / Phishing"
        action = "Block or move to spam immediately"
        confidence = min(0.99, 0.65 + score / 100)
    elif score >= 22:
        category = "Suspicious / Needs Review"
        action = "Quarantine for manual review"
        confidence = min(0.95, 0.55 + score / 100)
    else:
        category = "Likely Safe"
        action = "Allow to inbox"
        confidence = max(0.65, 0.75 - score / 100)

    return {
        **state,
        "score": score,
        "matched_rules": matched_rules if matched_rules else ["No strong spam rules matched"],
        "category": category,
        "suggested_action": action,
        "confidence": round(confidence, 2)
    }


def optional_web_search_node(state: SpamState) -> SpamState:
    """
    Optional evidence gathering for suspicious domains or urgent scam phrases.
    Does not decide classification.
    """
    search_mode = state.get("search_mode", "Auto")
    category = state.get("category", "Likely Safe")
    urls = state.get("extracted_urls", [])

    if search_mode == "Off":
        return {**state, "web_evidence": "Web search disabled."}

    should_search = False
    query = None

    if search_mode == "On":
        should_search = True
    elif search_mode == "Auto" and category != "Likely Safe":
        should_search = True

    if should_search:
        if urls:
            query = f"is domain scam {domain_from_url(urls[0])}"
        else:
            # take a small suspicious chunk
            suspicious_terms = state.get("matched_rules", [])[:2]
            query = "email scam " + " ".join(suspicious_terms)

        evidence = serper_search(query)
        return {**state, "web_evidence": evidence}

    return {**state, "web_evidence": "No web enrichment needed."}


def llm_explainer_node(state: SpamState) -> SpamState:
    """
    LLM explains the rule-based result in professional language.
    It does NOT classify the email.
    """
    prompt = f"""
You are a cybersecurity product assistant.

You are NOT deciding classification.
Your task is only to explain the already computed result clearly.

Input:
Category: {state.get('category')}
Confidence: {state.get('confidence')}
Score: {state.get('score')}
Matched Rules:
{chr(10).join('- ' + r for r in state.get('matched_rules', []))}
Web Evidence:
{state.get('web_evidence', '')}
Suggested Action: {state.get('suggested_action')}

Write:
1. A short executive explanation
2. Key reasons in bullet style
3. A final recommendation in one line

Keep it concise, professional, and product-style.
"""

    try:
        resp = llm.invoke(prompt)
        explanation = resp.content if hasattr(resp, "content") else str(resp)
    except Exception as e:
        explanation = f"LLM explanation unavailable: {str(e)}"

    return {**state, "explanation": explanation}


def formatter_node(state: SpamState) -> SpamState:
    urls = state.get("extracted_urls", [])
    report = f"""
# Spam Filter Result

## Classification
**Category:** {state.get('category')}
**Confidence:** {state.get('confidence')}
**Risk Score:** {state.get('score')}/100
**Suggested Action:** {state.get('suggested_action')}

## Matched Rules
{chr(10).join([f"- {x}" for x in state.get("matched_rules", [])])}

## Extracted URLs
{chr(10).join([f"- {u}" for u in urls]) if urls else "- No URLs found"}

## Web Evidence
{state.get('web_evidence', 'No evidence')}

## LLM Explanation
{state.get('explanation', '')}
    """.strip()

    return {**state, "final_report": report}

In [6]:

# =========================
# 6. Use LangGraph framework
# =========================
builder = StateGraph(SpamState)

builder.add_node("preprocess", preprocess_node)
builder.add_node("reflex_rule_engine", reflex_rule_engine_node)
builder.add_node("optional_web_search", optional_web_search_node)
builder.add_node("llm_explainer", llm_explainer_node)
builder.add_node("formatter", formatter_node)

builder.set_entry_point("preprocess")
builder.add_edge("preprocess", "reflex_rule_engine")
builder.add_edge("reflex_rule_engine", "optional_web_search")
builder.add_edge("optional_web_search", "llm_explainer")
builder.add_edge("llm_explainer", "formatter")
builder.add_edge("formatter", END)

spam_graph = builder.compile()

In [7]:



# =========================
# 7. Inference function
# =========================
def run_spam_filter(
    subject: str,
    sender: str,
    body: str,
    strictness: str,
    search_mode: str
):
    if not body.strip() and not subject.strip():
        return (
            "Please enter email subject or body.",
            "N/A",
            0,
            "No action",
            "[]"
        )

    state = {
        "subject": subject,
        "sender": sender,
        "email_text": body,
        "strictness": strictness,
        "search_mode": search_mode
    }

    result = spam_graph.invoke(state)

    category = result.get("category", "Unknown")
    confidence = result.get("confidence", 0)
    score = result.get("score", 0)
    action = result.get("suggested_action", "No action")
    report = result.get("final_report", "")
    rules_json = json.dumps(result.get("matched_rules", []), indent=2)

    return report, category, score, action, rules_json


In [8]:
# =========================
# 8. Example emails
# =========================
EXAMPLES = {
    "Phishing Mail": {
        "subject": "Urgent: Verify Your Bank Account Now",
        "sender": "security-update@bank-alerts.xyz",
        "body": """Dear Customer,
Your bank account has been suspended due to unusual activity.
Verify your account immediately by clicking here:
http://secure-bank-alert.xyz/login
Failure to act now will result in permanent suspension.
Thanks."""
    },
    "Promotional Mail": {
        "subject": "Exclusive Deal - 70% Discount Today Only!!!",
        "sender": "promo@shopping-mail.com",
        "body": """Congratulations!
Claim now and enjoy a special promotion.
Buy now, shop now, limited time offer!!!
Visit: https://best-deals.click"""
    },
    "Safe Work Mail": {
        "subject": "Project Review Meeting Tomorrow",
        "sender": "manager@company.com",
        "body": """Hi team,
Please join the project review meeting tomorrow at 11 AM.
Agenda includes sprint review, blockers, and next steps.
Regards,
Manager"""
    }
}

def load_example(example_name):
    ex = EXAMPLES.get(example_name, {"subject": "", "sender": "", "body": ""})
    return ex["subject"], ex["sender"], ex["body"]

In [ ]:



# =========================
# 9. Professional Gradio UI
# =========================
CUSTOM_CSS = """
body {
    background: linear-gradient(135deg, #f8fbff, #eef7f2);
}

.gradio-container {
    max-width: 1400px !important;
    margin: auto;
}

.hero-wrap {
    border-radius: 18px;
    padding: 18px 22px;
    background: linear-gradient(90deg, #eef7ff, #f6fff7);
    border: 1px solid #d9e8ff;
    box-shadow: 0 8px 24px rgba(0,0,0,0.06);
    overflow: hidden;
}

.hero-title {
    font-size: 34px;
    font-weight: 800;
    color: #184e77;
    margin-bottom: 4px;
}

.hero-subtitle {
    font-size: 14px;
    color: #4d6b86;
    margin-bottom: 12px;
}

.marquee {
    width: 100%;
    overflow: hidden;
    white-space: nowrap;
    border-radius: 10px;
    background: linear-gradient(90deg, #dff6e8, #e6f0ff);
    border: 1px solid #cfe3ff;
    padding: 10px 0;
}

.marquee span {
    display: inline-block;
    padding-left: 100%;
    animation: marquee 18s linear infinite;
    color: #0b6e4f;
    font-weight: 700;
    font-size: 18px;
}

@keyframes marquee {
    0%   { transform: translateX(0%); }
    100% { transform: translateX(-100%); }
}

.soft-card {
    border-radius: 16px !important;
    border: 1px solid #dfe9f3 !important;
    box-shadow: 0 6px 18px rgba(0,0,0,0.05) !important;
}

.footer-note {
    color: #5b7083;
    font-size: 13px;
}
"""

theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="emerald",
    neutral_hue="slate"
)

with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="Spam Filter") as demo:
    gr.HTML("""
    <div class="hero-wrap">
        <div class="hero-title">Spam Filter</div>
        <div class="hero-subtitle">Simple Reflex AI Agent using LangGraph + Groq + Serper</div>
        <div class="marquee">
            <span>Spam Filter by Dr. Arpit Yadav • Spam Filter by Dr. Arpit Yadav • Spam Filter by Dr. Arpit Yadav</span>
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown("### Email Input")

            example_choice = gr.Dropdown(
                choices=list(EXAMPLES.keys()),
                value="Phishing Mail",
                label="Load Example Email",
                interactive=True
            )

            subject = gr.Textbox(
                label="Email Subject",
                placeholder="Enter subject line...",
                lines=2,
                elem_classes=["soft-card"]
            )

            sender = gr.Textbox(
                label="Sender Email / Sender Name",
                placeholder="e.g. noreply@company.com",
                lines=1,
                elem_classes=["soft-card"]
            )

            body = gr.Textbox(
                label="Email Body",
                placeholder="Paste email content here...",
                lines=14,
                elem_classes=["soft-card"]
            )

        with gr.Column(scale=2):
            gr.Markdown("### Settings")

            strictness = gr.Dropdown(
                choices=["Low", "Medium", "High"],
                value="Medium",
                label="Filtering Strictness",
                interactive=True
            )

            search_mode = gr.Dropdown(
                choices=["Auto", "On", "Off"],
                value="Auto",
                label="Web Search Enrichment",
                interactive=True
            )

            run_btn = gr.Button("Run Spam Filter", variant="primary")
            clear_btn = gr.Button("Clear", variant="secondary")

            gr.Markdown("""
            **How this works**
            - Rule-based simple reflex classification
            - Optional Serper web evidence
            - Groq LLM for explanation only
            """)

    with gr.Row():
        category = gr.Textbox(label="Category", elem_classes=["soft-card"])
        score = gr.Number(label="Risk Score", elem_classes=["soft-card"])
        action = gr.Textbox(label="Suggested Action", elem_classes=["soft-card"])

    with gr.Row():
        report = gr.Markdown(label="Detailed Report")
        rules_json = gr.Code(label="Matched Rules (JSON)", language="json")

    example_choice.change(
        fn=load_example,
        inputs=example_choice,
        outputs=[subject, sender, body]
    )

    run_btn.click(
        fn=run_spam_filter,
        inputs=[subject, sender, body, strictness, search_mode],
        outputs=[report, category, score, action, rules_json]
    )

    clear_btn.click(
        fn=lambda: ("", "", "", "Medium", "Auto", "", "", 0, "", "[]"),
        inputs=[],
        outputs=[subject, sender, body, strictness, search_mode, report, category, score, action, rules_json]
    )

    gr.Markdown(
        "<div class='footer-note'>Professional product-style demo for Google Colab • LangGraph workflow • Groq explanation layer • Serper enrichment</div>"
    )

demo.launch(debug=True, share=True)

/tmp/ipykernel_1235/1745481512.py:78: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="Spam Filter") as demo:
/tmp/ipykernel_1235/1745481512.py:78: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="Spam Filter") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5dca2f3684fee55bbf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
